In [ ]:
import corner

import matplotlib.pyplot as plt
import numpy as np

import torch
import h5py

from hubersed.paths import *

In [ ]:
DATA_PATH = PATHS["DATA"]
RESULTS_PATH = PATHS["RESULTS"]

In [ ]:
with h5py.File(DATA_PATH / "prospector_noise_spec_cue_15latent_snr3.h5", "r") as f:
    prospector_latents = f["latents"][:]
    prospector_norms = f["A"][:]
    prospector_specs = f["specs"][:]
    prospector_zs = f["zs"][:]

In [ ]:
with h5py.File(DATA_PATH / "spender_spec_15latent_snr3.h5", "r") as f:
    desi_latents = f["latents"][:]
    desi_norms = f["A"][:]
    desi_specs = f["specs"][:]
    desi_zs = f["zs"][:]

    desi_tids = f["target_ids"][:]

In [ ]:
# load outlier idx
desi_outlier_tids = torch.load(
    RESULTS_PATH / "desi_outliers_flow_nsf_15latent_snr3.pt",
    map_location="cpu",
    mmap=True,
    weights_only=False,
)["outlier_target_ids"]


desi_outlier_idx = np.isin(desi_tids, desi_outlier_tids)

In [ ]:
wave = np.linspace(3600, 9824, desi_specs.shape[1])

In [ ]:
print(f"There are {len(desi_outlier_tids)} outliers in the DESI latent space.")

In [ ]:
fig = plt.figure(figsize=(12, 12), dpi=150)

# prospector (background) — light gray
fig = corner.corner(
    prospector_latents,
    bins=50,
    plot_contours=False,
    fill_contours=False,
    plot_datapoints=True,
    plot_density=False,
    data_kwargs={"ms": 0.5, "alpha": 0.01, "color": "#999999"},
    hist_kwargs={"color": "#999999", "alpha": 0.5, "density": True},
    fig=fig,
)

# desi (middle layer)
fig = corner.corner(
    desi_latents,
    bins=50,
    plot_contours=False,
    fill_contours=False,
    plot_datapoints=True,
    plot_density=False,
    data_kwargs={"ms": 0.5, "alpha": 0.01, "color": "#1f77b4"},
    hist_kwargs={"color": "#1f77b4", "alpha": 0.5, "density": True},
    fig=fig,
)

# outliers ON TOP — bigger, more opaque
fig = corner.corner(
    desi_latents[desi_outlier_idx],
    bins=50,
    plot_contours=False,
    fill_contours=False,
    plot_datapoints=True,
    plot_density=False,
    data_kwargs={"ms": 2, "alpha": 0.3, "color": "#e31a1c"},
    hist_kwargs={"color": "#e31a1c", "alpha": 0.5, "density": True},
    fig=fig,
)

from matplotlib.lines import Line2D

fig.legend(
    handles=[
        Line2D([], [], color="#999999", marker="s", ls="", label="Prospector"),
        Line2D([], [], color="#1f77b4", marker="s", ls="", label="DESI"),
        Line2D([], [], color="#e31a1c", marker="o", ls="", label="Outliers"),
    ],
    loc="upper right",
    fontsize=24,
    frameon=False,
)

In [ ]:
## From parameter_file.py

# DESI Spectra
import pickle

# from hubersed.prospector.utils import load_lines
from prospect.sources import FastStepBasis


def build_sps():
    return FastStepBasis()


sps = build_sps()
# EM_LINES_A = load_lines()['emission']['wave_vac']
fsps_waves = sps.ssp.emline_wavelengths
fsps_optical = fsps_waves[(fsps_waves > 3600) & (fsps_waves < 9824)]

OUTLIERS_IDX = torch.load(
    RESULTS_PATH / "desi_outliers.pt", map_location="cpu", mmap=True
)["outlier_indices"]


def get_outlier_info(idx, streaming=True):
    """
    Get information about a specific outlier.

    Parameters
    ----------
    idx : int
        Index of the outlier to retrieve (0 to len(OUTLIERS_IDX)-1)

    Returns
    -------
    spec : np.ndarray
        The observed spectrum for the outlier, in the original units (flambda).
    unc : np.ndarray
        The uncertainty on the observed spectrum, in the original units (flambda).
    redshift : float
        The redshift of the outlier.
    mask : np.ndarray
        A boolean array indicating which pixels are valid (True) or should be masked (False).
    id : int
        The original ID of the spectrum in the DESI dataset, for reference.
    """

    chunk_size = 1024
    chunk_indices = OUTLIERS_IDX // chunk_size
    chunk_files = [
        DATA_PATH / "desi_spectra" / f"DESIchunk1024_{i}.pkl" for i in chunk_indices
    ]
    chunk_indices_in_chunk = OUTLIERS_IDX % chunk_size

    chunk_file = chunk_files[idx]
    idx_in_chunk = chunk_indices_in_chunk[idx]

    if not streaming:
        with open(chunk_file, "rb") as f:
            s, w, z, id, norm, *_ = pickle.load(f)
    else:
        with hffs.open(
            f"buckets/nikhil0504/hubersed-data/desi_spectra/{chunk_file.name}", "rb"
        ) as f:
            s, w, z, id, norm, *_ = pickle.load(f)

    # correct for normalization
    s = s * norm[:, None]
    w = w / norm[:, None] ** 2
    mask = np.isfinite(s) & np.isfinite(w) & (w > 0)

    # convert to numpy arrays if not already from torch tensors
    if isinstance(s, torch.Tensor):
        s = s.cpu().numpy()
    if isinstance(w, torch.Tensor):
        w = w.cpu().numpy()
    if isinstance(z, torch.Tensor):
        z = z.cpu().numpy()
    if isinstance(mask, torch.Tensor):
        mask = mask.cpu().numpy()
    if isinstance(id, torch.Tensor):
        id = id.cpu().numpy()

    return (
        s[idx_in_chunk],
        w[idx_in_chunk],
        z[idx_in_chunk],
        mask[idx_in_chunk],
        id[idx_in_chunk],
    )


def mask_spectral_lines(
    wave_obs, mask, z, line_waves=fsps_optical, halfwidth_kms=500.0
):
    """Mask spectral lines using a velocity-based window.

    Parameters
    ----------
    wave_obs : np.ndarray
        Observed wavelength array [Å].
    mask : np.ndarray
        Boolean array; True = good pixel, False = masked.
    z : float
        Redshift of the object.
    line_waves : np.ndarray
        Rest-frame vacuum wavelengths of lines to mask [Å].
    halfwidth_kms : float, optional
        Half-width of mask window in km/s. Default is 500 km/s.

    Returns
    -------
    np.ndarray
        Updated boolean mask.
    """
    c_kms = 299792.458
    mask = mask.copy()
    wave_rest = wave_obs / (1.0 + z)

    for line in line_waves:
        dwave = line * halfwidth_kms / c_kms
        mask &= np.abs(wave_rest - line) > dwave

    return mask

In [ ]:
# load spender model
from spender.data import desi
from spender import load_model

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

inst = desi.DESI().float().to(device)

# Build wave_rest according to selected mode
model = load_model(
    str(DATA_PATH / "spender_asc_run_6latent_zmax.pt"),
    inst,
    map_location="cpu",
    weights_only=False,
).float()
model = model.to(device)
model.eval()

In [ ]:
wave_obs = inst._wave_obs

In [ ]:
# CORRECTED masking: in-place continuum interpolation (length stays 7781),
# normalized input (same representation that produced desi_latents), one chunk load each.
from collections import defaultdict

wave = wave_obs.numpy()
chunk_size = 1024
gidx = (
    desi_outlier_idx.numpy()
    if hasattr(desi_outlier_idx, "numpy")
    else np.asarray(desi_outlier_idx)
)
N = len(gidx)
desi_outlier_latents_em_masked = torch.zeros(
    (N, desi_latents.shape[1]), dtype=torch.float32
)

# group outlier positions by chunk
by_chunk = defaultdict(list)
for pos in range(N):
    g = int(gidx[pos])
    by_chunk[g // chunk_size].append((pos, g % chunk_size))

for ci, items in by_chunk.items():
    with open(DATA_PATH / "desi_spectra" / f"DESIchunk1024_{ci}.pkl", "rb") as f:
        s_b, w_b, z_b, *_ = pickle.load(f)
    s_b = (
        s_b.numpy() if hasattr(s_b, "numpy") else np.asarray(s_b)
    )  # NORMALIZED spec (as stored = as used for desi_latents)
    z_b = z_b.numpy() if hasattr(z_b, "numpy") else np.asarray(z_b)
    for pos, r in items:
        s = s_b[r].astype(np.float64).copy()
        z = float(z_b[r])
        keep = mask_spectral_lines(
            wave,
            np.ones_like(s, dtype=bool),
            z,
            line_waves=fsps_optical,
            halfwidth_kms=500.0,
        )
        # replace line-window flux by linear interpolation of the surrounding continuum (length preserved)
        if (~keep).any() and keep.sum() > 1:
            s[~keep] = np.interp(wave[~keep], wave[keep], s[keep])
        st = torch.tensor(s, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            desi_outlier_latents_em_masked[pos] = model.encode(st).squeeze().cpu()
    print(f"chunk {ci}: {len(items)} outliers done", end="\r", flush=True)
print("\ndone:", N, "outliers re-encoded with emission lines interpolated out")

In [ ]:
# Isolation Forest for outlier detection
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
p_l_scaled = scaler.fit_transform(prospector_latents)
s_l_scaled = scaler.transform(desi_outlier_latents_em_masked.cpu())

iso = IsolationForest(
    n_estimators=300,
    max_samples=2048,  # subsampling makes it fast and robust
    contamination="auto",
    n_jobs=-1,
    random_state=0,
)
iso.fit(p_l_scaled)

scores_desi = iso.decision_function(s_l_scaled)
scores_prospector = iso.decision_function(p_l_scaled)


# threshold for outliers, worse than 0.1% in the prospector distribution
threshold = torch.quantile(torch.tensor(scores_prospector), 0.001)
outlier_mask = torch.tensor(scores_desi) <= threshold
outlier_idx_em_masked = torch.where(outlier_mask)[0]

In [ ]:
# save these em outliers idx
torch.save(
    {"outlier_indices": outlier_idx_em_masked},
    RESULTS_PATH / "desi_outliers_em_masked.pt",
)

In [ ]:
len(outlier_idx_em_masked)

In [ ]:
C_CGS = 2.99792458e10  # cm/s
FNU_PER_MAGGIE = 3631e-23  # erg/s/cm^2/Hz


def flambda_to_maggies(wave_A, flambda):
    # flambda: erg/s/cm^2/Å ; wave_A: Å
    flambda_cgs = (
        flambda * 1e-17
    )  # DESI spectra are in 1e-17 erg/s/cm^2/Å, convert to cgs
    return flambda_cgs * (wave_A**2) * 1e-8 / C_CGS / FNU_PER_MAGGIE

In [ ]:
fig = plt.figure(figsize=(12, 12), dpi=150)

# prospector (background) — light gray
fig = corner.corner(
    prospector_latents.numpy(),
    bins=50,
    plot_contours=False,
    fill_contours=False,
    plot_datapoints=True,
    plot_density=False,
    data_kwargs={"ms": 0.5, "alpha": 0.01, "color": "#999999"},
    hist_kwargs={"color": "#999999", "alpha": 0.5, "density": True},
    fig=fig,
)

# # desi (middle layer)
# fig = corner.corner(
#     desi_latents.numpy(),
#     bins=50,
#     plot_contours=False, fill_contours=False,
#     plot_datapoints=True, plot_density=False,
#     data_kwargs={"ms": 0.5, "alpha": 0.01, "color": "#1f77b4"},
#     fig=fig,
# )

# outliers ON TOP — bigger, more opaque
fig = corner.corner(
    desi_latents[desi_outlier_idx],
    bins=50,
    plot_contours=False,
    fill_contours=False,
    plot_datapoints=True,
    plot_density=False,
    data_kwargs={"ms": 2, "alpha": 0.3, "color": "#e31a1c"},
    hist_kwargs={"color": "#e31a1c", "alpha": 0.5, "density": True},
    fig=fig,
)

# outlier latents with em lines masked out — ON TOP of the outliers, different color
small_data = desi_outlier_latents_em_masked[outlier_mask].numpy()
axes = np.array(fig.axes).reshape(6, 6)

for i in range(6):
    for j in range(i):
        axes[i, j].plot(
            small_data[:, j],
            small_data[:, i],
            "o",
            ms=4,
            alpha=0.8,
            color="#336a25",
            zorder=10,
        )

fig = corner.corner(
    desi_outlier_latents_em_masked[~outlier_mask].numpy(),
    bins=50,
    plot_contours=False,
    fill_contours=False,
    plot_datapoints=True,
    plot_density=False,
    data_kwargs={"ms": 2, "alpha": 0.3, "color": "#a90783"},
    hist_kwargs={"color": "#a90783", "alpha": 0.5, "density": True},
    fig=fig,
)

from matplotlib.lines import Line2D

fig.legend(
    handles=[
        Line2D([], [], color="#999999", marker="s", ls="", label="Prospector"),
        # Line2D([], [], color="#1f77b4", marker="s", ls="", label="DESI"),
        Line2D([], [], color="#e31a1c", marker="o", ls="", label="Outliers"),
        Line2D(
            [], [], color="#336a25", marker="o", ls="", label="Outliers after EM Masked"
        ),
        Line2D(
            [], [], color="#a90783", marker="o", ls="", label="Non-Outliers (EM Masked)"
        ),
    ],
    loc="upper right",
    fontsize=24,
    frameon=False,
)

## Control 1 — is the masking *specific* to outliers?

Mask emission lines on a random sample of **non-outliers**, re-encode, and measure how far their
latents move vs how far the outliers moved. If outliers move far (manifold edge -> inside) while
non-outliers barely budge, the emission-line signal is specifically what made the outliers anomalous.
We also check that masked non-outliers do not themselves turn into outliers under the same IsolationForest.


In [ ]:
# Control 1: latent displacement from masking, non-outliers vs outliers (uses iso/scaler/threshold from above)
rng = np.random.default_rng(0)
nonout = np.setdiff1d(np.arange(desi_latents.shape[0]), gidx)
samp = rng.choice(nonout, size=2000, replace=False)

ctrl_masked = torch.zeros((len(samp), desi_latents.shape[1]), dtype=torch.float32)
bc = defaultdict(list)
for pos, g in enumerate(samp):
    bc[int(g) // chunk_size].append((pos, int(g) % chunk_size))
for ci, items in bc.items():
    with open(DATA_PATH / "desi_spectra" / f"DESIchunk1024_{ci}.pkl", "rb") as f:
        s_b, w_b, z_b, *_ = pickle.load(f)
    s_b = s_b.numpy() if hasattr(s_b, "numpy") else np.asarray(s_b)
    z_b = z_b.numpy() if hasattr(z_b, "numpy") else np.asarray(z_b)
    for pos, r in items:
        s = s_b[r].astype(np.float64).copy()
        z = float(z_b[r])
        keep = mask_spectral_lines(
            wave,
            np.ones_like(s, dtype=bool),
            z,
            line_waves=fsps_optical,
            halfwidth_kms=500.0,
        )
        if (~keep).any() and keep.sum() > 1:
            s[~keep] = np.interp(wave[~keep], wave[keep], s[keep])
        with torch.no_grad():
            ctrl_masked[pos] = (
                model.encode(
                    torch.tensor(s, dtype=torch.float32, device=device).unsqueeze(0)
                )
                .squeeze()
                .cpu()
            )

# displacement in latent space (masked - original)
d_nonout = np.linalg.norm(ctrl_masked.numpy() - desi_latents[samp], axis=1)
d_out = np.linalg.norm(
    desi_outlier_latents_em_masked.numpy() - desi_latents[gidx], axis=1
)
print(
    f"latent shift |masked-orig|:  non-outliers med={np.median(d_nonout):.3f}   outliers med={np.median(d_out):.3f}"
)

# do masked non-outliers become outliers under the SAME iso?
sc_ctrl = iso.decision_function(scaler.transform(ctrl_masked.numpy()))
n_new = int((torch.tensor(sc_ctrl) <= threshold).sum())
print(
    f"masked non-outliers now flagged outlier: {n_new}/{len(samp)} ({100 * n_new / len(samp):.2f}%)"
)

fig, ax = plt.subplots(figsize=(6, 4))
b = np.linspace(0, np.percentile(np.concatenate([d_nonout, d_out]), 99), 50)
ax.hist(
    d_nonout, bins=b, density=True, histtype="step", color="k", label="non-outliers"
)
ax.hist(
    d_out,
    bins=b,
    density=True,
    histtype="stepfilled",
    color="red",
    alpha=0.4,
    label="outliers",
)
ax.set_xlabel("|latent shift| from masking")
ax.set_ylabel("density")
ax.legend()
plt.show()

## Control 2 — apples-to-apples (mask Prospector too)

The outlier test so far compares **masked** (continuum-only) DESI latents against **unmasked** Prospector
latents. To attribute the signal cleanly, mask emission lines on the Prospector mocks as well, refit the
IsolationForest on continuum-only Prospector latents, and re-score the continuum-only DESI outliers.

- still ~2 outliers  -> the discrepancy was in the **lines** (masking both removes it).
- many outliers again -> a **continuum-level** mock-vs-data mismatch remains, independent of lines.


In [ ]:
# Control 2: mask Prospector lines too, refit iso on continuum-only Prospector, re-score continuum-only DESI
rng = np.random.default_rng(1)
ps_all = (
    prospector_specs.numpy()
    if hasattr(prospector_specs, "numpy")
    else np.asarray(prospector_specs)
)
pz_all = (
    prospector_zs.numpy()
    if hasattr(prospector_zs, "numpy")
    else np.asarray(prospector_zs)
)
wave_p = np.linspace(
    3600.0, 9824.0, ps_all.shape[1]
)  # match Prospector grid length (may be 7780, not 7781)
sub = rng.choice(ps_all.shape[0], size=min(50000, ps_all.shape[0]), replace=False)
prosp_masked = torch.zeros((len(sub), desi_latents.shape[1]), dtype=torch.float32)

B = 512
for k in range(0, len(sub), B):
    blk = sub[k : k + B]
    arr = ps_all[blk].astype(np.float64).copy()
    for m, gi in enumerate(blk):
        z = float(pz_all[gi])
        keep = mask_spectral_lines(
            wave_p,
            np.ones(arr.shape[1], dtype=bool),
            z,
            line_waves=fsps_optical,
            halfwidth_kms=500.0,
        )
        if (~keep).any() and keep.sum() > 1:
            arr[m, ~keep] = np.interp(wave_p[~keep], wave_p[keep], arr[m, keep])
    with torch.no_grad():
        prosp_masked[k : k + B] = model.encode(
            torch.tensor(arr, dtype=torch.float32, device=device)
        ).cpu()
    print(f"prospector masked {min(k + B, len(sub))}/{len(sub)}", end="\r", flush=True)
print()

scaler2 = StandardScaler()
pm = scaler2.fit_transform(prosp_masked.numpy())
iso2 = IsolationForest(
    n_estimators=300, max_samples=2048, contamination="auto", n_jobs=-1, random_state=0
)
iso2.fit(pm)
thr2 = torch.quantile(torch.tensor(iso2.decision_function(pm)), 0.001)
sc_d = iso2.decision_function(
    scaler2.transform(desi_outlier_latents_em_masked.cpu().numpy())
)
n2 = int((torch.tensor(sc_d) <= thr2).sum())
print(f"masked-DESI outliers still flagged vs MASKED Prospector: {n2}/{len(gidx)}")